# NeurIPS 2026 Smart Buildings — Version 20: Sliding-mode / nonlinear HVAC control

**Independent direction:** Control from the current comfort-boundary error and a clipped, filtered temperature-velocity estimate. Version 7's mixing relation is retained only for AHU physics coordination; Version 20's **control law** is a nonlinear saturated sliding-surface reaching rule, not V7's load/headroom-only feed-forward dispatch.

## Algorithm
1. Decode metadata-indexed zone and outdoor observations into Kelvin.
2. Form `s_hot = T - high + λ · trend` and `s_cold = low - T - λ · trend`, with `λ = 1.8` five-minute steps, filtered/clipped trend, and an extreme-violation override.
3. Map positive surfaces into smooth commands using a finite-width saturation layer plus a nonlinear square-root term. Do not use a discontinuous sign controller.
4. Use separate cooling-coil/damper and reheat-valve channels. Hysteresis controls AHU start/stop; a simple mixing estimate gates economizer recirculation.
5. Reset the trend and AHU latches between episodes; emit bounded float32/int32 actions using the official inference server.

**Hypothesis:** directional derivative feedback may reject changing outdoor/occupancy disturbances more quickly than V7 while smooth boundary layers limit chatter. **Not yet validated:** No local simulator or public leaderboard score has been obtained. No formal stability or performance guarantee is claimed because the actual plant's input gains/disturbances are uncertain.

## Evaluation plan
Run the same 30 public training scenarios as V7 and V14. Compare score, energy, carbon, occupied-zone tail comfort violations (>5 K), AHU switch count, damper/valve total variation, and runtime. Split by building for tuning; retain unseen building/weather validation. Report any regressions rather than inferring improvement from static tests.

## Notebook use
The final cell calls `SmartBuildingsInferenceServer(TransferPolicy).run()`. No external data, files, online optimization, packages, or state from previous buildings are required.


In [ ]:
"""Version 20: boundary-layer sliding-mode control of smart-building HVAC.

Per-zone sliding surfaces combine temperature-band error with filtered recent
temperature velocity. Saturated nonlinear reaching laws govern VAV airflow,
AHU cooling, and local reheat. Hysteretic AHU enable prevents hard switching,
and the thermal mixing relation is used only to coordinate coil/economizer.

An unknown nonlinear building plant prevents a formal stability guarantee.
This is an empirical robust-control experiment, O(AHUs + zones) per step.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Sliding-surface robust HVAC dispatch with metadata-indexed actions.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical bounds and finite-width switching layer.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15

    # s_hot = T - high + horizon * dT; s_cold = low - T - horizon * dT.
    # dT is measured in Kelvin per five-minute decision.
    HORIZON_STEPS = 1.8
    TREND_ALPHA = 0.35
    MAX_STEP_CHANGE_K = 0.45
    SURFACE_LAYER_K = 0.70
    SWITCH_ON_K = 0.17
    SWITCH_OFF_K = 0.06

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        outside_norm = meta.get("observation_normalization_metadata", {}).get(
            "outside_air_temperature_sensor", {})
        self.outside_mean = float(outside_norm.get("sample_mean", np.nan))
        self.outside_std = float(outside_norm.get("sample_std", np.nan))
        if not (np.isfinite(self.outside_mean) and np.isfinite(self.outside_std)
                and self.outside_std > 0.0):
            self.outside_mean = self.outside_std = np.nan

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """Forget prior observations and AHU states at episode boundaries."""
        self._previous_zone_temp = {}
        self._filtered_zone_trend = {}
        self._ahu_running = [False] * len(self.ahus)

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode Kelvin and estimate a clipped, filtered five-minute trend.

        Differencing temperature (not the setpoint error) avoids setpoint
        schedule transitions producing a derivative kick.
        """
        index = zone["temperature"]
        raw = self._value(obs, index)
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            self._previous_zone_temp.pop(index, None)
            self._filtered_zone_trend.pop(index, None)
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback

        previous = self._previous_zone_temp.get(index)
        delta = (self._clamp(temperature - previous,
                             -self.MAX_STEP_CHANGE_K, self.MAX_STEP_CHANGE_K)
                 if previous is not None else 0.0)
        prior = self._filtered_zone_trend.get(index, 0.0)
        trend = self.TREND_ALPHA * delta + (1.0 - self.TREND_ALPHA) * prior
        self._previous_zone_temp[index] = temperature
        self._filtered_zone_trend[index] = trend

        hot = max(0.0, temperature - high)
        cold = max(0.0, low - temperature)
        hot_surface = temperature - high + self.HORIZON_STEPS * trend
        cold_surface = low - temperature - self.HORIZON_STEPS * trend
        # Do not drop extreme comfort penalties merely because T is improving.
        hot_surface = max(hot_surface, 0.40 * max(0.0, hot - 3.0))
        cold_surface = max(cold_surface, 0.40 * max(0.0, cold - 3.0))
        return {
            "actions": zone, "temp": temperature, "hot": hot, "cold": cold,
            "hot_surface": hot_surface, "cold_surface": cold_surface,
            "cool_signal": self._reaching_signal(hot_surface),
            "heat_signal": self._reaching_signal(cold_surface),
        }

    @classmethod
    def _reaching_signal(cls, surface):
        """Finite-boundary-layer saturated reaching law, with nonlinear gain.

        This is sliding-mode-inspired, not a claimed exact model-canceling
        equivalent control law for the unknown simulator dynamics.
        """
        s = max(0.0, float(surface))
        linear = min(1.0, s / cls.SURFACE_LAYER_K)
        nonlinear = np.sqrt(s / (1.2 + s)) if s > 0.0 else 0.0
        return cls._clamp(0.65 * linear + 0.35 * nonlinear, 0.0, 1.0)

    def _outside_air(self, obs, ahu):
        """Denormalize outdoor temperature before any Kelvin comparison."""
        value = self._value(obs, ahu["outdoor"])
        if not np.isfinite(value) or not np.isfinite(self.outside_std):
            return float("nan")
        kelvin = value * self.outside_std + self.outside_mean
        return kelvin if 210.0 <= kelvin <= 330.0 else float("nan")

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Set nonlinear cooling demand and check economizer feasibility.

        The mixing equation is a coordination estimate; the sliding surface
        rather than a predictive plant model chooses conditioning intensity.
        """
        peak_cooling = max(z["cool_signal"] for z in zones)
        severe_cooling = max(z["hot"] for z in zones)
        return_k = float(np.mean([z["temp"] for z in zones]))
        if peak_cooling <= 0.02:
            coil_k = self.MAX_SUPPLY_K
        else:
            coil_k = self._clamp(
                299.0 - 11.0 * np.sqrt(peak_cooling)
                - 0.55 * min(severe_cooling, 6.0),
                self.MIN_SUPPLY_K, 298.0)

        if not np.isfinite(outside_k):
            outside_k = return_k
        recirc = self.MAX_RETURN_FRACTION
        if (peak_cooling > 0.02
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            min_mix = (self.MIN_ECONOMIZER_RECIRC * return_k
                       + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix <= coil_k:
                desired = (coil_k - outside_k) / (return_k - outside_k)
                recirc = self._clamp(
                    desired, self.MIN_ECONOMIZER_RECIRC,
                    self.MAX_RETURN_FRACTION)
        mixed = recirc * return_k + (1.0 - recirc) * outside_k
        return coil_k, recirc, min(mixed, coil_k)

    def _control_zone_boxes(self, statuses, delivered, water_demands, continuous):
        """Translate reaching signals to bounded damper and reheat commands."""
        total_damper, peak_damper = 0.0, 0.0
        for state in statuses:
            zone = state["actions"]
            cool, heat = state["cool_signal"], state["heat_signal"]
            if cool > 0.10 and cool >= heat:
                headroom = max(2.5, state["temp"] - delivered)
                damper = self._clamp(
                    self.MIN_VAV_FRACTION
                    + 0.82 * np.sqrt(cool) * np.sqrt(5.0 / headroom),
                    self.MIN_VAV_FRACTION, 0.98)
            elif heat > 0.10 and zone["valve"] is not None:
                damper = self._clamp(
                    0.18 + 0.48 * np.sqrt(heat), 0.18, 0.82)
                valve = self._clamp(0.08 + 0.82 * np.sqrt(heat), 0.0, 1.0)
                continuous[zone["valve"]] = self._fraction_action(valve)
                water_demands.append((max(state["cold"], state["cold_surface"]),
                                      valve))
            else:
                damper = self.MIN_VAV_FRACTION
            total_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        return total_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Hysteretic AHU run decision followed by nonlinear VAV dispatch."""
        if not statuses:
            self._ahu_running[ahu["run"]] = False
            return
        reachable = max(
            max(z["hot_surface"],
                z["cold_surface"] if z["actions"]["valve"] is not None else -np.inf)
            for z in statuses)
        running = self._ahu_running[ahu["run"]]
        should_run = (reachable >= self.SWITCH_OFF_K if running
                      else reachable >= self.SWITCH_ON_K)
        self._ahu_running[ahu["run"]] = bool(should_run)
        if not should_run:
            return

        coil_k, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(coil_k)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            loading = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (loading - self.MIN_VAV_FRACTION) / 0.65, 0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Observe -> update sliding surfaces -> dispatch -> sanitize actions."""
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoor_values = [], []
        for ahu in self.ahus:
            statuses = [self._read_zone(obs, z, fallback)
                        for z in ahu["zones"]]
            statuses = [z for z in statuses if z is not None]
            outdoor = self._outside_air(obs, ahu)
            if np.isfinite(outdoor):
                outdoor_values.append(outdoor)
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(
            hot_water_demands, outdoor_values, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# Official Kaggle evaluation gateway; not run by local unit tests.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
